# Consolidated Report — Review Notes

Companion to `consolidated_report.ipynb`. That notebook produces the paper's tables and figures;
this one says what is **wrong, missing, or still owed** before any of it is transferred to LaTeX.

Nothing here belongs in the final document. Tables are written to `resources/notes/`, which the
LaTeX build file deliberately excludes.

Run `consolidated_report.ipynb` first — the manifest and figure audit below read what it wrote.

In [1]:
import sys
import pathlib

sys.path.insert(0, str(pathlib.Path.cwd()))

import pandas as pd

import report_lib as R
from report_lib import (ALPHA, DASH, MARKETS, MARKET_LABEL, STAR_LEVELS,
                        emit, integer, num, pct, read, sig)

# review tables go to resources/notes/, never to resources/tables/
for _stale in R.NOTES.glob("*"):
    if _stale.is_file():
        _stale.unlink()

NOTE = dict(outdir=R.NOTES)

print(f"results   : {R.RESULTS}")
print(f"notes out : {R.NOTES}")
print(f"alpha     : {ALPHA:.2f}   stars: "
      + ", ".join(f"{m} for p < {lv:g}" for lv, m in STAR_LEVELS))

results   : /Users/lap14821-local/Documents/Software Development/worldquant_uni/fscore_capstone/results
notes out : /Users/lap14821-local/Documents/Software Development/worldquant_uni/fscore_capstone/consolidated_report/resources/notes
alpha     : 0.05   stars: * for p < 0.05


---
## 1. Open items

Ten findings, each with the evidence that produced it. The first five change how the numbers may
be *described* in the paper; the rest are gaps in what has been produced.

In [2]:
issues = pd.DataFrame(
    [
        ("N1", "Japan's study window is two formation years (2023, 2024)",
         "japan_diagnostics.csv; japan_fullperiod_feasibility.csv reports "
         "“no point-in-time statements” for 2005--2022",
         "No 13-year window is common to all three markets. §8's headline "
         "cross-country table cannot be described as covering 2012--2024."),
        ("N2", "Japan's 2023 basket holds 9 names, not K",
         "japan_diagnostics.csv (k = 9 in 2023, 30 in 2024)",
         "Half of Japan's headline result rests on a 9-name portfolio. The "
         "Effective N of 19.5 is the average of the two years, not a basket size."),
        ("N3", "The eligible universe is capped at the top 150 names by dollar volume",
         "pipeline.py:105,141; the cap binds in every market-year except "
         "Japan 2023 (118) and US 2010 (121)",
         "After the 40% B/M cut the null draws from 60 names, so a random "
         "basket of K = 30 shares half its names with the F-Score basket. The "
         "Monte Carlo test has little power by construction."),
        ("N4", "The high-B/M set is 40% of the cap, not 40% of the market",
         "high_bm_subset takes a relative quantile (loaders.py:78); "
         "value_set = 60 in every year where universe = 150, without exception",
         "The count cannot respond to the market: a value drawdown does not widen "
         "it. Uncapped, 40% would be 121--342 names in Vietnam. The B/M threshold "
         "is also a moving target, differing by year and by market."),
        ("N5", "Monte Carlo N differs by strategy (1000 EW / 300 GMV)",
         "*_mc_placement.csv, n_draws column",
         "§6 requires one N for all three headline country tests. GMV p-values "
         "are quoted on a coarser grid than EW p-values."),
        ("N6", "The US and Japan fundamentals caches are absent",
         "data/ holds only vietnam_*; loaders.py:47 reads "
         "data/{market}_fundamentals.csv for both markets",
         "The US and Japan headline results cannot be regenerated as they stand. "
         "scripts/fetch_us_japan.py must rebuild the caches first, and refreshed "
         "Yahoo data may not reproduce the committed numbers."),
        ("N7", "Maximum weight and non-zero name counts are not exported",
         "*_summary.csv carries effective_n only",
         "P24 asks for both. Those two columns stay as — in Table 8.1 until the "
         "pipeline writes them."),
        ("N8", "Drawdown and annual-weight figures are not produced",
         "results/figures/ holds NAV, MC placement and F-Score distribution only",
         "P17, P20 and P23 can be filled only in part; see the figure audit below."),
        ("N9", "The $-100\\%$ delisting robustness case was not run",
         "no results file for it",
         "P28 must either be run or state plainly that it was not."),
        ("N10", "Sector-capped GMV exists in all three markets but is placed nowhere",
         "fscore_GMVsec rows in every *_summary.csv",
         "P10 and P29 both wait on this decision: headline or robustness."),
    ],
    columns=["#", "Open item", "Evidence", "Effect on the paper"],
).set_index("#")

emit(issues,
     name="note_1_open_items",
     number="Note 1",
     title="Open items blocking the transfer to LaTeX",
     placeholder="review checklist",
     section="internal",
     index_header="#",
     notes=("Compiled from the results files and the implementation, not from the draft. "
            "N1--N5 constrain what the numbers may be said to show; N6--N10 are gaps in "
            "what has been produced."),
     **NOTE)

**Note 1. Open items blocking the transfer to LaTeX**  
<sub>internal &middot; fills review checklist &middot; `resources/notes/note_1_open_items.tex`</sub>

#,Open item,Evidence,Effect on the paper
N1,"Japan's study window is two formation years (2023, 2024)",japan_diagnostics.csv; japan_fullperiod_feasibility.csv reports “no point-in-time statements” for 2005--2022,No 13-year window is common to all three markets. §8's headline cross-country table cannot be described as covering 2012--2024.
N2,"Japan's 2023 basket holds 9 names, not K","japan_diagnostics.csv (k = 9 in 2023, 30 in 2024)","Half of Japan's headline result rests on a 9-name portfolio. The Effective N of 19.5 is the average of the two years, not a basket size."
N3,The eligible universe is capped at the top 150 names by dollar volume,"pipeline.py:105,141; the cap binds in every market-year except Japan 2023 (118) and US 2010 (121)","After the 40% B/M cut the null draws from 60 names, so a random basket of K = 30 shares half its names with the F-Score basket. The Monte Carlo test has little power by construction."
N4,"The high-B/M set is 40% of the cap, not 40% of the market","high_bm_subset takes a relative quantile (loaders.py:78); value_set = 60 in every year where universe = 150, without exception","The count cannot respond to the market: a value drawdown does not widen it. Uncapped, 40% would be 121--342 names in Vietnam. The B/M threshold is also a moving target, differing by year and by market."
N5,Monte Carlo N differs by strategy (1000 EW / 300 GMV),"*_mc_placement.csv, n_draws column",§6 requires one N for all three headline country tests. GMV p-values are quoted on a coarser grid than EW p-values.
N6,The US and Japan fundamentals caches are absent,data/ holds only vietnam_*; loaders.py:47 reads data/{market}_fundamentals.csv for both markets,"The US and Japan headline results cannot be regenerated as they stand. scripts/fetch_us_japan.py must rebuild the caches first, and refreshed Yahoo data may not reproduce the committed numbers."
N7,Maximum weight and non-zero name counts are not exported,*_summary.csv carries effective_n only,P24 asks for both. Those two columns stay as — in Table 8.1 until the pipeline writes them.
N8,Drawdown and annual-weight figures are not produced,"results/figures/ holds NAV, MC placement and F-Score distribution only","P17, P20 and P23 can be filled only in part; see the figure audit below."
N9,The $-100\%$ delisting robustness case was not run,no results file for it,P28 must either be run or state plainly that it was not.
N10,Sector-capped GMV exists in all three markets but is placed nowhere,fscore_GMVsec rows in every *_summary.csv,P10 and P29 both wait on this decision: headline or robustness.


<sub>*Notes.* Compiled from the results files and the implementation, not from the draft. N1--N5 constrain what the numbers may be said to show; N6--N10 are gaps in what has been produced.</sub>

,Open item,Evidence,Effect on the paper
#,,,
N1,Japan's study window is two formation years (2...,japan_diagnostics.csv; japan_fullperiod_feasib...,No 13-year window is common to all three marke...
N2,"Japan's 2023 basket holds 9 names, not K","japan_diagnostics.csv (k = 9 in 2023, 30 in 2024)",Half of Japan's headline result rests on a 9-n...
N3,The eligible universe is capped at the top 150...,"pipeline.py:105,141; the cap binds in every ma...",After the 40% B/M cut the null draws from 60 n...
N4,"The high-B/M set is 40% of the cap, not 40% of...",high_bm_subset takes a relative quantile (load...,The count cannot respond to the market: a valu...
N5,Monte Carlo N differs by strategy (1000 EW / 3...,"*_mc_placement.csv, n_draws column",§6 requires one N for all three headline count...
N6,The US and Japan fundamentals caches are absent,data/ holds only vietnam_*; loaders.py:47 read...,The US and Japan headline results cannot be re...
N7,Maximum weight and non-zero name counts are no...,*_summary.csv carries effective_n only,P24 asks for both. Those two columns stay as —...
N8,Drawdown and annual-weight figures are not pro...,"results/figures/ holds NAV, MC placement and F...","P17, P20 and P23 can be filled only in part; s..."
N9,The $-100\%$ delisting robustness case was not...,no results file for it,P28 must either be run or state plainly that i...


---
## 2. Figure audit

Which of the figures §7 asks for exist, and which do not.

In [3]:
rows = {}
for market, src_name, dest_name, what in R.FIG_PLAN:
    section = "§9.1" if "grid" in src_name else f"§7.{MARKETS.index(market) + 1}"
    exists = (R.FIGURES / dest_name).exists()
    rows[dest_name] = {"Market": MARKET_LABEL[market], "Shows": what,
                       "Section": section,
                       "Status": "copied" if exists else sig("source missing", True)}

# the figures the draft asks for that nothing in the repository produces
for market in MARKETS:
    n = MARKETS.index(market) + 1
    for what in R.FIG_NOT_PRODUCED:
        rows[f"(none) {MARKET_LABEL[market]} — {what.lower()}"] = {
            "Market": MARKET_LABEL[market], "Shows": what,
            "Section": f"§7.{n}", "Status": sig("not produced", True)}

figures = pd.DataFrame(rows).T[["Market", "Shows", "Section", "Status"]]
figures.index.name = "File in resources/figures"

emit(figures,
     name="note_2_figure_audit",
     number="Note 2",
     title="Figure audit",
     placeholder="P17, P20, P23",
     section="§7, §9.1",
     index_header="File in resources/figures",
     notes=("The draft asks for four figures per country: cumulative performance, drawdown, "
            "the Monte Carlo null distribution, and annual weights. The first and third exist. "
            "A standalone drawdown chart and an annual-weight chart are produced by no script "
            "in the repository — either add them to the plotting step or drop them from §7."),
     **NOTE)

**Note 2. Figure audit**  
<sub>§7, §9.1 &middot; fills P17, P20, P23 &middot; `resources/notes/note_2_figure_audit.tex`</sub>

File in resources/figures,Market,Shows,Section,Status
fig_7_1a_us_cumulative.png,United States,Cumulative performance,§7.1,copied
fig_7_1b_us_mc_null.png,United States,Monte Carlo null distribution,§7.1,copied
fig_7_1c_us_fscore_distribution.png,United States,F-Score distribution,§7.1,copied
fig_7_1d_us_cumulative_full.png,United States,"Cumulative performance, full window",§7.1,copied
fig_7_2a_japan_cumulative.png,Japan,Cumulative performance,§7.2,copied
fig_7_2b_japan_mc_null.png,Japan,Monte Carlo null distribution,§7.2,copied
fig_7_2c_japan_fscore_distribution.png,Japan,F-Score distribution,§7.2,copied
fig_7_2d_japan_cumulative_full.png,Japan,"Cumulative performance, full window",§7.2,copied
fig_7_3a_vietnam_cumulative.png,Vietnam,Cumulative performance,§7.3,copied
fig_7_3b_vietnam_mc_null.png,Vietnam,Monte Carlo null distribution,§7.3,copied


<sub>*Notes.* The draft asks for four figures per country: cumulative performance, drawdown, the Monte Carlo null distribution, and annual weights. The first and third exist. A standalone drawdown chart and an annual-weight chart are produced by no script in the repository — either add them to the plotting step or drop them from §7.</sub>

,Market,Shows,Section,Status
File in resources/figures,,,,
fig_7_1a_us_cumulative.png,United States,Cumulative performance,§7.1,copied
fig_7_1b_us_mc_null.png,United States,Monte Carlo null distribution,§7.1,copied
fig_7_1c_us_fscore_distribution.png,United States,F-Score distribution,§7.1,copied
fig_7_1d_us_cumulative_full.png,United States,"Cumulative performance, full window",§7.1,copied
fig_7_2a_japan_cumulative.png,Japan,Cumulative performance,§7.2,copied
fig_7_2b_japan_mc_null.png,Japan,Monte Carlo null distribution,§7.2,copied
fig_7_2c_japan_fscore_distribution.png,Japan,F-Score distribution,§7.2,copied
fig_7_2d_japan_cumulative_full.png,Japan,"Cumulative performance, full window",§7.2,copied
fig_7_3a_vietnam_cumulative.png,Vietnam,Cumulative performance,§7.3,copied


---
## 3. Placeholders still owed to a human

Everything a script cannot fill: prose, or a decision.

In [4]:
prose = pd.DataFrame(
    [
        ("P01", "§1", "One-paragraph summary of the unified findings",
         "Write after the numbers are accepted; do not reuse the M6 preliminary text"),
        ("P02", "§2", "RMT / covariance-cleaning references",
         "The implementation follows Lopez de Prado (2020); add the citation"),
        ("P06", "§3.7", "Operational definition of a stale or non-trading security",
         "The code exits at the last tradable price with volume; write that definition down"),
        ("P07", "§4.2", "Field hierarchy used to identify equity issuance",
         "See results/eq_offer_headline.csv and eq_offer_sensitivity.csv"),
        ("P09", "§5.3", "Eigenvalue-cleaning and detoning equations",
         "Transcribe from construction/weights.py:16--62; note detoning is off for any inverse"),
        ("P25", "§8", "Cross-country interpretation",
         "Must separate the selection effect, the construction effect, and data-coverage "
         "differences; N1 constrains what can be claimed"),
        ("P29", "§9.4", "Whether the sector-constrained GMV subsection stays",
         "The variant is produced in all three markets; the decision is still open (N10)"),
        ("P30", "§9.5", "Whether the economic-regime subsection stays",
         "Not rerun under the unified design; rerun or drop"),
        ("P31", "§10", "Discussion of where the F-Score signal is strongest",
         "Do not infer from the superseded broad-universe runs; use Tables 7.1b--7.3b"),
        ("P33", "§12", "Final empirical conclusion",
         "Four separate statements: selection, construction, robustness, limitations"),
        ("P34", "References", "RMT and optimisation references",
         "Same citation set as P02 and P09"),
    ],
    columns=["Placeholder", "Section", "What it asks for", "What to do"],
).set_index("Placeholder")

emit(prose,
     name="note_3_prose_placeholders",
     number="Note 3",
     title="Placeholders that need prose or a decision, not a table",
     placeholder="P01, P02, P06, P07, P09, P25, P29, P30, P31, P33, P34",
     section="all",
     index_header="Placeholder",
     notes="Every other placeholder in the draft is filled by a table or figure in the report.",
     **NOTE)

**Note 3. Placeholders that need prose or a decision, not a table**  
<sub>all &middot; fills P01, P02, P06, P07, P09, P25, P29, P30, P31, P33, P34 &middot; `resources/notes/note_3_prose_placeholders.tex`</sub>

Placeholder,Section,What it asks for,What to do
P01,§1,One-paragraph summary of the unified findings,Write after the numbers are accepted; do not reuse the M6 preliminary text
P02,§2,RMT / covariance-cleaning references,The implementation follows Lopez de Prado (2020); add the citation
P06,§3.7,Operational definition of a stale or non-trading security,The code exits at the last tradable price with volume; write that definition down
P07,§4.2,Field hierarchy used to identify equity issuance,See results/eq_offer_headline.csv and eq_offer_sensitivity.csv
P09,§5.3,Eigenvalue-cleaning and detoning equations,Transcribe from construction/weights.py:16--62; note detoning is off for any inverse
P25,§8,Cross-country interpretation,"Must separate the selection effect, the construction effect, and data-coverage differences; N1 constrains what can be claimed"
P29,§9.4,Whether the sector-constrained GMV subsection stays,The variant is produced in all three markets; the decision is still open (N10)
P30,§9.5,Whether the economic-regime subsection stays,Not rerun under the unified design; rerun or drop
P31,§10,Discussion of where the F-Score signal is strongest,Do not infer from the superseded broad-universe runs; use Tables 7.1b--7.3b
P33,§12,Final empirical conclusion,"Four separate statements: selection, construction, robustness, limitations"


<sub>*Notes.* Every other placeholder in the draft is filled by a table or figure in the report.</sub>

,Section,What it asks for,What to do
Placeholder,,,
P01,§1,One-paragraph summary of the unified findings,Write after the numbers are accepted; do not r...
P02,§2,RMT / covariance-cleaning references,The implementation follows Lopez de Prado (202...
P06,§3.7,Operational definition of a stale or non-tradi...,The code exits at the last tradable price with...
P07,§4.2,Field hierarchy used to identify equity issuance,See results/eq_offer_headline.csv and eq_offer...
P09,§5.3,Eigenvalue-cleaning and detoning equations,Transcribe from construction/weights.py:16--62...
P25,§8,Cross-country interpretation,"Must separate the selection effect, the constr..."
P29,§9.4,Whether the sector-constrained GMV subsection ...,The variant is produced in all three markets; ...
P30,§9.5,Whether the economic-regime subsection stays,Not rerun under the unified design; rerun or drop
P31,§10,Discussion of where the F-Score signal is stro...,Do not infer from the superseded broad-univers...


---
## 4. Build manifest

What the report actually wrote, read back off disk.

In [5]:
# Read from resources/ rather than from R.MANIFEST, so this notebook can be
# run on its own and still reports what is really there.
tex = sorted(R.TABLES.glob("*.tex"))
png = sorted(R.FIGURES.glob("*.png"))
all_tables = R.RES / "_all_tables.tex"

rows = {}
for f in tex:
    body = f.read_text(encoding="utf-8")
    caption = next((l.split("{", 1)[1].rsplit("}", 1)[0]
                    for l in body.splitlines() if l.startswith(r"\caption{")), "")
    rows[f.name] = {
        "Title": caption,
        "In _all_tables.tex": ("yes" if all_tables.exists()
                               and f.stem in all_tables.read_text(encoding="utf-8")
                               else sig("no", True)),
        "Size": f"{f.stat().st_size:,} B",
    }

manifest = pd.DataFrame(rows).T[["Title", "In _all_tables.tex", "Size"]]
manifest.index.name = "File in resources/tables"

emit(manifest,
     name="note_4_build_manifest",
     number="Note 4",
     title="Build manifest: what the report wrote",
     placeholder="build record",
     section="internal",
     index_header="File in resources/tables",
     notes=(f"{len(tex)} tables and {len(png)} figures on disk. A table marked “no” exists but "
            "is not pulled into the LaTeX build — either it was renamed or the report was not "
            "re-run after it was added."),
     **NOTE)

**Note 4. Build manifest: what the report wrote**  
<sub>internal &middot; fills build record &middot; `resources/notes/note_4_build_manifest.tex`</sub>

File in resources/tables,Title,In _all_tables.tex,Size
table_11_1_data_quality.tex,"Data source and coverage, by country",yes,"2,170 B"
table_11_2_exclusions.tex,"Source firm-years against scoreable firm-years, by country",yes,"1,286 B"
table_11_3_feasibility.tex,"Formation-year feasibility, by country",yes,"1,643 B"
table_3_1_design_decisions.tex,Unified research-design decisions for the cross-country analysis,yes,"2,987 B"
table_7_0_sample_windows.tex,"Formation-year windows actually produced, by market",yes,"1,118 B"
table_7_1a_us_funnel.tex,United States: formation-year counts at every logged stage,yes,"1,662 B"
table_7_1b_us_performance_headline.tex,United States: performance over the headline window,yes,"2,635 B"
table_7_1c_us_performance_full.tex,United States: performance over the full available window,yes,"2,443 B"
table_7_2a_japan_funnel.tex,Japan: formation-year counts at every logged stage,yes,"1,133 B"
table_7_2b_japan_performance_headline.tex,Japan: performance over the headline window,yes,"2,596 B"


<sub>*Notes.* 19 tables and 15 figures on disk. A table marked “no” exists but is not pulled into the LaTeX build — either it was renamed or the report was not re-run after it was added.</sub>

,Title,In _all_tables.tex,Size
File in resources/tables,,,
table_11_1_data_quality.tex,"Data source and coverage, by country",yes,"2,170 B"
table_11_2_exclusions.tex,Source firm-years against scoreable firm-years...,yes,"1,286 B"
table_11_3_feasibility.tex,"Formation-year feasibility, by country",yes,"1,643 B"
table_3_1_design_decisions.tex,Unified research-design decisions for the cros...,yes,"2,987 B"
table_7_0_sample_windows.tex,"Formation-year windows actually produced, by m...",yes,"1,118 B"
table_7_1a_us_funnel.tex,United States: formation-year counts at every ...,yes,"1,662 B"
table_7_1b_us_performance_headline.tex,United States: performance over the headline w...,yes,"2,635 B"
table_7_1c_us_performance_full.tex,United States: performance over the full avail...,yes,"2,443 B"
table_7_2a_japan_funnel.tex,Japan: formation-year counts at every logged s...,yes,"1,133 B"


In [6]:
# Inputs the report expected but could not find. An empty result means every
# table in the report was built from files that exist.
missing = sorted(set(R.MISSING))
if missing:
    df = pd.DataFrame({"Missing input": missing}).set_index("Missing input")
    emit(df, name="note_5_missing_inputs", number="Note 5",
         title="Expected inputs that were not found", placeholder="build record",
         section="internal", index_header="Missing input",
         notes="Each left a — in a report table, or a figure uncopied.", **NOTE)
else:
    print("No missing inputs: every table in the report was built from files that exist.")

No missing inputs: every table in the report was built from files that exist.


---
## What to do next

1. **N1--N5 first.** They change what the numbers may be said to show, not merely whether the
   set is complete. In particular, the US F-Score equal-weight portfolio sits at $p = 0.527$ on
   the Sharpe ratio against its matched random null — a coin flip, and the honest headline for
   that market.
2. **Settle three design decisions** before the LaTeX pass, because each changes which rows
   survive: the headline GMV specification, one Monte Carlo $N$, and whether the sector-capped
   variant is headline or robustness.
3. **Decide the star convention.** The report uses the single 5% level fixed in §6. The
   three-level convention common in the published F-Score papers is one line in `report_lib.py`
   (`STAR_LEVELS`), but adopting it means amending §6 — declaring one level in advance and then
   reporting three is the specification-searching that section exists to prevent.
4. **Then copy `resources/` into the LaTeX project** and add `\input{resources/_all_tables}`.
   `resources/notes/` is not part of that build.